# EDA conjunto — Data4Cyber (S0–S5 + S1-alt)

Análisis exploratorio sobre las ** tablas de proceso** del release, unidas en un solo
`DataFrame` con la columna `escenario`.

**Cómo se ejecuta.** La celda 0 resuelve sola la ruta de datos: monta Drive si corre en
Colab y cae a `datasets/` si corre local. No hay rutas absolutas en ninguna otra celda.
***Se utiliza hasta el escenario S5 ya que hay datos sin comprender en el escenario S6***

In [42]:
# === 0. Configuración y carga de los escenarios ===

%matplotlib inline
import os


from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)


from google.colab import drive
drive.mount("/content/gdrive/")
BASE =Path("/content/gdrive/MyDrive/TrabajoFinal/datasets/")


import pandas as pd

dfs = {
    #"S0":     pd.read_csv(BASE / "datasetS0.csv"),
    "S1":     pd.read_csv(BASE / "datasetS1.csv"),
    "S1-alt": pd.read_csv(BASE / "datasetS1alt.csv"),
    "S2":     pd.read_csv(BASE / "datasetS2.csv"),
    #"S3":     pd.read_csv(BASE / "datasetS3.csv"),
    "S4":     pd.read_csv(BASE / "datasetS4.csv"),
    "S5":     pd.read_csv(BASE / "datasetS5.csv"),
    "S6":     pd.read_csv(BASE / "datasetS6.csv"),
}
df = pd.concat(dfs.values(), ignore_index=True)

Drive already mounted at /content/gdrive/; to attempt to forcibly remount, call drive.mount("/content/gdrive/", force_remount=True).


In [ ]:
from IPython.display import display
def mostrar_dfs(dfs,n):
    for k, df in dfs.items():
        print(f"=== {k} ===")
        display(df.head(n))

mostrar_dfs(dfs,5)

In [44]:
# --- agrupación de columnas: dispositivo, tipo, fase ---
SIG_TIPO = {
    "uln1": "Tensión fase-neutro", "uln2": "Tensión fase-neutro", "uln3": "Tensión fase-neutro",
    "ull12": "Tensión fase-fase", "ull23": "Tensión fase-fase", "ull31": "Tensión fase-fase",
    "iln1": "Corriente de fase", "iln2": "Corriente de fase", "iln3": "Corriente de fase",
    "i_sum3": "Corriente de neutro",
    "pln1": "Potencia activa", "pln2": "Potencia activa", "pln3": "Potencia activa", "p_sum3": "Potencia activa",
    "sln1": "Potencia aparente", "sln2": "Potencia aparente", "sln3": "Potencia aparente", "s_sum3": "Potencia aparente",
    "qln1": "Potencia reactiva", "qln2": "Potencia reactiva", "qln3": "Potencia reactiva", "q_sum3": "Potencia reactiva",
    "cos_phi1": "Factor de potencia", "cos_phi2": "Factor de potencia", "cos_phi3": "Factor de potencia",
    "freq": "Frecuencia",
    "phase_seq": "Secuencia de fases",
    "realtime": "Marca de tiempo del equipo",
}
UNIDAD = {
    "Tensión fase-neutro": "V", "Tensión fase-fase": "V", "Corriente de fase": "A",
    "Corriente de neutro": "A", "Potencia activa": "W", "Potencia aparente": "VA",
    "Potencia reactiva": "var", "Factor de potencia": "—", "Frecuencia": "Hz",
    "Secuencia de fases": "—", "Marca de tiempo del equipo": "s",
    "SOC": "fracción 0-1", "Señal de precio": "EUR/kWh", "Contexto de escenario": "—",
    "Control / etiquetas": "—", "Desconocido": "?",
}
FASE = {"ull12": "L1-L2", "ull23": "L2-L3", "ull31": "L3-L1"}

def partir(c):
    if "." in c:
        dev, sig = c.split(".", 1)
    else:
        dev, sig = "(sin prefijo)", c
    return dev, sig

def tipo_de(dev, sig):
    if dev == "(sin prefijo)":
        return "Control / etiquetas"
    if dev.startswith("Battery-SOC"):
        return "SOC"
    if dev == "MQTT-Price-Signal":
        return "Señal de precio"
    if dev == "Profile":
        return "Contexto de escenario"
    return SIG_TIPO.get(sig, "Desconocido")

def fase_de(sig):
    if sig in FASE:
        return FASE[sig]
    if sig.endswith("sum3"):
        return "trifásica"
    if sig.endswith("1"):
        return "L1"
    if sig.endswith("2"):
        return "L2"
    if sig.endswith("3"):
        return "L3"
    return "—"

reg = []
for c in df.columns:
    dev, sig = partir(c)
    t = tipo_de(dev, sig)
    reg.append({"columna": c, "dispositivo": dev, "señal": sig,
                "tipo": t, "unidad": UNIDAD.get(t, "?"), "fase": fase_de(sig)})
cols = pd.DataFrame(reg)

print("columnas:", len(cols), "| dispositivos:", cols.dispositivo.nunique(), "| tipos:", cols.tipo.nunique())
print("\n--- por dispositivo ---")
por_dev = (cols.groupby("dispositivo")
           .agg(columnas=("columna", "size"), tipos=("tipo", lambda s: ", ".join(sorted(set(s)))))
           .sort_values("columnas", ascending=False))
display(por_dev)

columnas: 156 | dispositivos: 10 | tipos: 15

--- por dispositivo ---


,columnas,tipos
dispositivo,,
BSS-Inverter-AC-Meter,28,"Corriente de fase, Corriente de neutro, Factor..."
Load-B-AC-Meter,28,"Corriente de fase, Corriente de neutro, Factor..."
Substation-AC-Meter,28,"Corriente de fase, Corriente de neutro, Factor..."
Load-A-AC-Meter,28,"Corriente de fase, Corriente de neutro, Factor..."
PV-Inverter-AC-Meter,28,"Corriente de fase, Corriente de neutro, Factor..."
Profile,8,Contexto de escenario
(sin prefijo),4,Control / etiquetas
MQTT-Price-Signal,2,Señal de precio
Battery-SOC-Reference,1,SOC


## 2. Datos Nulos

En este release **una celda vacía no es un cero ni un `NaN` imputado: es ausencia de valor de origen**.

In [45]:
# --- vacíos por columna y por fila ---

num = df.select_dtypes(include=[np.number])
no_num = [c for c in df.columns if c not in num.columns]
miss = df.isna()

print(f"columnas numéricas: {len(num.columns)} | no numéricas: {len(no_num)}")
if no_num:
    print("no numéricas:", no_num)

print(f"celdas vacías: {int(miss.values.sum()):,} de {miss.size:,} ({miss.values.mean()*100:.2f} %)")
print(f"filas con algún vacío: {int((miss.sum(axis=1) > 0).sum()):,} de {len(miss):,}")

por_col = miss.mean().mul(100).sort_values(ascending=False)
con_vacios = por_col[por_col > 0]
print(f"ncolumnas con al menos un vacío: {len(con_vacios)} de {len(por_col)}")
print("\ntop 15 columnas más incompletas (%):")
print(con_vacios.head(15).round(2).to_string())
#print(con_vacios.index.tolist())



columnas numéricas: 150 | no numéricas: 6
no numéricas: ['timestamp', 'attack_active', 'attack_phase', 'attack_phase_all', 'Profile.timestamp', 'MQTT-Price-Signal.event']
celdas vacías: 15,774 de 1,677,936 (0.94 %)
filas con algún vacío: 9,394 de 10,756
ncolumnas con al menos un vacío: 152 de 156

top 15 columnas más incompletas (%):
MQTT-Price-Signal.event                87.34
MQTT-Price-Signal.price_eur_per_kwh     5.01
Battery-SOC-Reported.soc_value          1.53
Profile.total_irradiance                1.50
Profile.load1_kw                        1.50
Profile.pv_input_kw                     1.50
Profile.load2_kw                        1.50
Profile.pv_active_kw                    1.50
Profile.pv_reactive_kvar                1.50
Profile.timestamp                       1.50
Profile.ambient_temperature             1.50
Battery-SOC-Reference.soc_value         0.45
BSS-Inverter-AC-Meter.qln3              0.29
BSS-Inverter-AC-Meter.realtime          0.29
BSS-Inverter-AC-Meter.s_sum3      

 # Eliminar columnas sin aporte
 por ahora vamos a descartar las columas que involucren marcas temporales y etapas del ataque ya que no estamos interezados en analizar estas caracteristicas.
"

In [46]:
def eliminar_columnas(df):
    df = df.drop(columns=["attack_phase", "attack_phase_all","timestamp"])
    df = df.drop(columns=[c for c in df.columns if "MQTT-Price-Signal" in c ], errors='ignore')
    df = df.drop(columns=[c for c in df.columns if "realtime" in c ])
    df = df.drop(columns=[c for c in df.columns if "phase_seq" in c ])
    df = df.drop(columns=[c for c in df.columns if "Profile" in c ])


    return df

dfs={k: eliminar_columnas(df) for k, df in dfs.items()}


# Imputacion de datos 

In [ ]:
dfs2={k:df[df.columns[df.isna().any()]] for k, df in dfs.items()}
mostrar_dfs(dfs2,3)

In [48]:
def imputar_filas_con_nulos(df):
    columnas_con_nulos = df.columns[df.isna().any()]
    for col in columnas_con_nulos:
        df[col] = df[col].fillna(df[col].mean())
    return df

dfs={k: imputar_filas_con_nulos(df) for k, df in dfs.items()}

# Split de Datos
Debido a la naturaleza temporal e interdependiente de las variables de telemetría ciberfísica, aplicar una división aleatoria convencional (*random train-test split*) introduce el riesgo de data leakage y overfitting, ya que el modelo tiende a memorizar tendencias continuas o correlaciones contiguas dentro de un mismo episodio operacional. Para capturar la dinámica secuencial de los sensores sin contaminar el entrenamiento, los datos se estructuraron mediante rolling windows de 20 pasos temporales hacia atrás. Asimismo, la validación del modelo se diseñó bajo un esquema de Leave-One-Scenario-Out(LOSO), lo que garantiza que el conjunto de prueba contenga únicamente eventos de ataque u operación completamente inéditos, evaluando con rigor la capacidad real de generalización del sistema ante situaciones no vistas

In [53]:
def separar_variables(df):
    dfX = df.drop(["attack_active"],axis=1)
    dfY = df["attack_active"]
    return dfX, dfY


dfsX, dfsY = {}, {}
for k, df in dfs.items():
    dfsX[k], dfsY[k] = separar_variables(df)

In [ ]:
mostrar_dfs(dfsX,3)
mostrar_dfs(dfsY,3)

In [50]:
# === 3. Aplanado de ventanas: 20 pasos x N canales por ventana ===

VENTANA = 20

# --- una fila por ventana: concatena los 20 pasos en orden temporal (t0c0..t0cN, t1c0..) ---
def aplanar_ventanas(dfX, y, ventana=VENTANA):
    """Devuelve la matriz aplanada y la etiqueta de la ultima fila de cada ventana."""
    X = dfX.to_numpy(dtype=np.float32)
    etiquetas = y.to_numpy()
    bloques = np.stack([X[i:i + ventana] for i in range(len(X) - ventana + 1)])
    columnas = [f"{canal}_t{paso}" for paso in range(ventana) for canal in dfX.columns]
    planas = bloques.reshape(len(bloques), -1)
    return pd.DataFrame(planas, columns=columnas), pd.Series(etiquetas[ventana - 1:], name="attack_active")

# --- se aplana escenario por escenario: ninguna ventana cruza el final de uno con el inicio del siguiente ---
dfsX_flat, dfsY_flat = {}, {}
for k in dfsX:
    dfsX_flat[k], dfsY_flat[k] = aplanar_ventanas(dfsX[k], dfsY[k])

# --- control de forma: columnas = canales x ventana, y ventanas = filas - 19 por escenario ---
primero = next(iter(dfsX_flat))
print("canales por paso:", dfsX[primero].shape[1])
print("columnas por ventana:", dfsX_flat[primero].shape[1])
print("ventanas por escenario:", {k: len(v) for k, v in dfsX_flat.items()})
print("ventanas totales:", sum(len(v) for v in dfsX_flat.values()))
print("positivos por escenario:", {k: int(v.sum()) for k, v in dfsY_flat.items()})

canales por paso: 132
columnas por ventana: 2640
ventanas por escenario: {'S1': 1779, 'S1-alt': 1773, 'S2': 1749, 'S4': 1779, 'S5': 1781, 'S6': 1781}
ventanas totales: 10642
positivos por escenario: {'S1': 913, 'S1-alt': 910, 'S2': 903, 'S4': 1257, 'S5': 1341, 'S6': 1350}


In [ ]:
mostrar_dfs(dfsX,3)

In [54]:
def rango_movil(df, ventana=120):
    ventana_rodante = df.rolling(ventana, min_periods=1)
    maximo = ventana_rodante.max()
    minimo = ventana_rodante.min()

    return (maximo - minimo).reindex(df.index)  


dfs2={k: rango_movil(df) for k, df in dfsX.items()}
mostrar_dfs(dfs2,8)


=== S1 ===


,BSS-Inverter-AC-Meter.cos_phi1,BSS-Inverter-AC-Meter.cos_phi2,BSS-Inverter-AC-Meter.cos_phi3,BSS-Inverter-AC-Meter.freq,BSS-Inverter-AC-Meter.i_sum3,BSS-Inverter-AC-Meter.iln1,BSS-Inverter-AC-Meter.iln2,BSS-Inverter-AC-Meter.iln3,BSS-Inverter-AC-Meter.p_sum3,BSS-Inverter-AC-Meter.pln1,BSS-Inverter-AC-Meter.pln2,BSS-Inverter-AC-Meter.pln3,BSS-Inverter-AC-Meter.q_sum3,BSS-Inverter-AC-Meter.qln1,BSS-Inverter-AC-Meter.qln2,BSS-Inverter-AC-Meter.qln3,BSS-Inverter-AC-Meter.s_sum3,BSS-Inverter-AC-Meter.sln1,BSS-Inverter-AC-Meter.sln2,BSS-Inverter-AC-Meter.sln3,BSS-Inverter-AC-Meter.ull12,BSS-Inverter-AC-Meter.ull23,BSS-Inverter-AC-Meter.ull31,BSS-Inverter-AC-Meter.uln1,BSS-Inverter-AC-Meter.uln2,BSS-Inverter-AC-Meter.uln3,Battery-SOC-Reference.soc_value,Battery-SOC-Reported.soc_value,Load-A-AC-Meter.cos_phi1,Load-A-AC-Meter.cos_phi2,...,PV-Inverter-AC-Meter.ull31,PV-Inverter-AC-Meter.uln1,PV-Inverter-AC-Meter.uln2,PV-Inverter-AC-Meter.uln3,Substation-AC-Meter.cos_phi1,Substation-AC-Meter.cos_phi2,Substation-AC-Meter.cos_phi3,Substation-AC-Meter.freq,Substation-AC-Meter.i_sum3,Substation-AC-Meter.iln1,Substation-AC-Meter.iln2,Substation-AC-Meter.iln3,Substation-AC-Meter.p_sum3,Substation-AC-Meter.pln1,Substation-AC-Meter.pln2,Substation-AC-Meter.pln3,Substation-AC-Meter.q_sum3,Substation-AC-Meter.qln1,Substation-AC-Meter.qln2,Substation-AC-Meter.qln3,Substation-AC-Meter.s_sum3,Substation-AC-Meter.sln1,Substation-AC-Meter.sln2,Substation-AC-Meter.sln3,Substation-AC-Meter.ull12,Substation-AC-Meter.ull23,Substation-AC-Meter.ull31,Substation-AC-Meter.uln1,Substation-AC-Meter.uln2,Substation-AC-Meter.uln3
0,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.0000,0.0,0.00000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
1,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.1458,0.0,0.00000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
2,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.1458,0.0,0.00000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
3,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.1458,0.0,0.00000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
4,1.279093,0.699096,0.245318,0.00087,0.017487,0.003971,0.006475,0.01016,44.130928,18.01724,16.741507,12.27452,114.146263,30.759619,45.87742,37.509227,17.044281,3.422142,7.479004,6.143112,0.006653,0.111694,0.102966,0.0298,0.003571,0.094391,0.1458,0.0,0.01734,0.000201,...,

=== S1-alt ===


,BSS-Inverter-AC-Meter.cos_phi1,BSS-Inverter-AC-Meter.cos_phi2,BSS-Inverter-AC-Meter.cos_phi3,BSS-Inverter-AC-Meter.freq,BSS-Inverter-AC-Meter.i_sum3,BSS-Inverter-AC-Meter.iln1,BSS-Inverter-AC-Meter.iln2,BSS-Inverter-AC-Meter.iln3,BSS-Inverter-AC-Meter.p_sum3,BSS-Inverter-AC-Meter.pln1,BSS-Inverter-AC-Meter.pln2,BSS-Inverter-AC-Meter.pln3,BSS-Inverter-AC-Meter.q_sum3,BSS-Inverter-AC-Meter.qln1,BSS-Inverter-AC-Meter.qln2,BSS-Inverter-AC-Meter.qln3,BSS-Inverter-AC-Meter.s_sum3,BSS-Inverter-AC-Meter.sln1,BSS-Inverter-AC-Meter.sln2,BSS-Inverter-AC-Meter.sln3,BSS-Inverter-AC-Meter.ull12,BSS-Inverter-AC-Meter.ull23,BSS-Inverter-AC-Meter.ull31,BSS-Inverter-AC-Meter.uln1,BSS-Inverter-AC-Meter.uln2,BSS-Inverter-AC-Meter.uln3,Battery-SOC-Reference.soc_value,Battery-SOC-Reported.soc_value,Load-A-AC-Meter.cos_phi1,Load-A-AC-Meter.cos_phi2,...,PV-Inverter-AC-Meter.ull31,PV-Inverter-AC-Meter.uln1,PV-Inverter-AC-Meter.uln2,PV-Inverter-AC-Meter.uln3,Substation-AC-Meter.cos_phi1,Substation-AC-Meter.cos_phi2,Substation-AC-Meter.cos_phi3,Substation-AC-Meter.freq,Substation-AC-Meter.i_sum3,Substation-AC-Meter.iln1,Substation-AC-Meter.iln2,Substation-AC-Meter.iln3,Substation-AC-Meter.p_sum3,Substation-AC-Meter.pln1,Substation-AC-Meter.pln2,Substation-AC-Meter.pln3,Substation-AC-Meter.q_sum3,Substation-AC-Meter.qln1,Substation-AC-Meter.qln2,Substation-AC-Meter.qln3,Substation-AC-Meter.s_sum3,Substation-AC-Meter.sln1,Substation-AC-Meter.sln2,Substation-AC-Meter.sln3,Substation-AC-Meter.ull12,Substation-AC-Meter.ull23,Substation-AC-Meter.ull31,Substation-AC-Meter.uln1,Substation-AC-Meter.uln2,Substation-AC-Meter.uln3
0,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.207608,0.517314,0.13445,0.002377,0.004369,0.006325,0.02539,0.007019,24.670556,6.934627,10.072766,7.663163,93.655321,19.329231,30.669046,43.657043,9.072784,1.501465,5.92131,1.650017,0.028503,0.042816,0.093384,0.054871,0.027786,0.018463,0.000000

=== S2 ===


,BSS-Inverter-AC-Meter.cos_phi1,BSS-Inverter-AC-Meter.cos_phi2,BSS-Inverter-AC-Meter.cos_phi3,BSS-Inverter-AC-Meter.freq,BSS-Inverter-AC-Meter.i_sum3,BSS-Inverter-AC-Meter.iln1,BSS-Inverter-AC-Meter.iln2,BSS-Inverter-AC-Meter.iln3,BSS-Inverter-AC-Meter.p_sum3,BSS-Inverter-AC-Meter.pln1,BSS-Inverter-AC-Meter.pln2,BSS-Inverter-AC-Meter.pln3,BSS-Inverter-AC-Meter.q_sum3,BSS-Inverter-AC-Meter.qln1,BSS-Inverter-AC-Meter.qln2,BSS-Inverter-AC-Meter.qln3,BSS-Inverter-AC-Meter.s_sum3,BSS-Inverter-AC-Meter.sln1,BSS-Inverter-AC-Meter.sln2,BSS-Inverter-AC-Meter.sln3,BSS-Inverter-AC-Meter.ull12,BSS-Inverter-AC-Meter.ull23,BSS-Inverter-AC-Meter.ull31,BSS-Inverter-AC-Meter.uln1,BSS-Inverter-AC-Meter.uln2,BSS-Inverter-AC-Meter.uln3,Battery-SOC-Reference.soc_value,Battery-SOC-Reported.soc_value,Load-A-AC-Meter.cos_phi1,Load-A-AC-Meter.cos_phi2,...,PV-Inverter-AC-Meter.ull31,PV-Inverter-AC-Meter.uln1,PV-Inverter-AC-Meter.uln2,PV-Inverter-AC-Meter.uln3,Substation-AC-Meter.cos_phi1,Substation-AC-Meter.cos_phi2,Substation-AC-Meter.cos_phi3,Substation-AC-Meter.freq,Substation-AC-Meter.i_sum3,Substation-AC-Meter.iln1,Substation-AC-Meter.iln2,Substation-AC-Meter.iln3,Substation-AC-Meter.p_sum3,Substation-AC-Meter.pln1,Substation-AC-Meter.pln2,Substation-AC-Meter.pln3,Substation-AC-Meter.q_sum3,Substation-AC-Meter.qln1,Substation-AC-Meter.qln2,Substation-AC-Meter.qln3,Substation-AC-Meter.s_sum3,Substation-AC-Meter.sln1,Substation-AC-Meter.sln2,Substation-AC-Meter.sln3,Substation-AC-Meter.ull12,Substation-AC-Meter.ull23,Substation-AC-Meter.ull31,Substation-AC-Meter.uln1,Substation-AC-Meter.uln2,Substation-AC-Meter.uln3
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000
4,0.499132,0.765428,0.871202,0.001881,0.057727,0.005558,0.012673,0.009739,33.473066,10.463104,12.689788,10.320174,70.022408,12.563803,7.266794,7.204634,4.106323,1.275711,3.051605,2.330444,0.143768,0.169098,0.059326,0.057404

... (2 outputs left out)

=== S5 ===


,BSS-Inverter-AC-Meter.cos_phi1,BSS-Inverter-AC-Meter.cos_phi2,BSS-Inverter-AC-Meter.cos_phi3,BSS-Inverter-AC-Meter.freq,BSS-Inverter-AC-Meter.i_sum3,BSS-Inverter-AC-Meter.iln1,BSS-Inverter-AC-Meter.iln2,BSS-Inverter-AC-Meter.iln3,BSS-Inverter-AC-Meter.p_sum3,BSS-Inverter-AC-Meter.pln1,BSS-Inverter-AC-Meter.pln2,BSS-Inverter-AC-Meter.pln3,BSS-Inverter-AC-Meter.q_sum3,BSS-Inverter-AC-Meter.qln1,BSS-Inverter-AC-Meter.qln2,BSS-Inverter-AC-Meter.qln3,BSS-Inverter-AC-Meter.s_sum3,BSS-Inverter-AC-Meter.sln1,BSS-Inverter-AC-Meter.sln2,BSS-Inverter-AC-Meter.sln3,BSS-Inverter-AC-Meter.ull12,BSS-Inverter-AC-Meter.ull23,BSS-Inverter-AC-Meter.ull31,BSS-Inverter-AC-Meter.uln1,BSS-Inverter-AC-Meter.uln2,BSS-Inverter-AC-Meter.uln3,Battery-SOC-Reference.soc_value,Battery-SOC-Reported.soc_value,Load-A-AC-Meter.cos_phi1,Load-A-AC-Meter.cos_phi2,...,PV-Inverter-AC-Meter.ull31,PV-Inverter-AC-Meter.uln1,PV-Inverter-AC-Meter.uln2,PV-Inverter-AC-Meter.uln3,Substation-AC-Meter.cos_phi1,Substation-AC-Meter.cos_phi2,Substation-AC-Meter.cos_phi3,Substation-AC-Meter.freq,Substation-AC-Meter.i_sum3,Substation-AC-Meter.iln1,Substation-AC-Meter.iln2,Substation-AC-Meter.iln3,Substation-AC-Meter.p_sum3,Substation-AC-Meter.pln1,Substation-AC-Meter.pln2,Substation-AC-Meter.pln3,Substation-AC-Meter.q_sum3,Substation-AC-Meter.qln1,Substation-AC-Meter.qln2,Substation-AC-Meter.qln3,Substation-AC-Meter.s_sum3,Substation-AC-Meter.sln1,Substation-AC-Meter.sln2,Substation-AC-Meter.sln3,Substation-AC-Meter.ull12,Substation-AC-Meter.ull23,Substation-AC-Meter.ull31,Substation-AC-Meter.uln1,Substation-AC-Meter.uln2,Substation-AC-Meter.uln3
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.003401,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.003401,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.003401,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.039733,0.139829,0.069832,0.001072,0.035015,0.020233,0.016092,0.019053,9.638258,2.56952,4.33364,2.735099,11.026039,9.244732,9.909983,8.128668,4.096313,2.346046,3.573448,1.823189,0.133057,0.154724,0.071625,0.051636,0.088608,0.067322,

=== S6 ===


,BSS-Inverter-AC-Meter.cos_phi1,BSS-Inverter-AC-Meter.cos_phi2,BSS-Inverter-AC-Meter.cos_phi3,BSS-Inverter-AC-Meter.freq,BSS-Inverter-AC-Meter.i_sum3,BSS-Inverter-AC-Meter.iln1,BSS-Inverter-AC-Meter.iln2,BSS-Inverter-AC-Meter.iln3,BSS-Inverter-AC-Meter.p_sum3,BSS-Inverter-AC-Meter.pln1,BSS-Inverter-AC-Meter.pln2,BSS-Inverter-AC-Meter.pln3,BSS-Inverter-AC-Meter.q_sum3,BSS-Inverter-AC-Meter.qln1,BSS-Inverter-AC-Meter.qln2,BSS-Inverter-AC-Meter.qln3,BSS-Inverter-AC-Meter.s_sum3,BSS-Inverter-AC-Meter.sln1,BSS-Inverter-AC-Meter.sln2,BSS-Inverter-AC-Meter.sln3,BSS-Inverter-AC-Meter.ull12,BSS-Inverter-AC-Meter.ull23,BSS-Inverter-AC-Meter.ull31,BSS-Inverter-AC-Meter.uln1,BSS-Inverter-AC-Meter.uln2,BSS-Inverter-AC-Meter.uln3,Battery-SOC-Reference.soc_value,Battery-SOC-Reported.soc_value,Load-A-AC-Meter.cos_phi1,Load-A-AC-Meter.cos_phi2,...,PV-Inverter-AC-Meter.ull31,PV-Inverter-AC-Meter.uln1,PV-Inverter-AC-Meter.uln2,PV-Inverter-AC-Meter.uln3,Substation-AC-Meter.cos_phi1,Substation-AC-Meter.cos_phi2,Substation-AC-Meter.cos_phi3,Substation-AC-Meter.freq,Substation-AC-Meter.i_sum3,Substation-AC-Meter.iln1,Substation-AC-Meter.iln2,Substation-AC-Meter.iln3,Substation-AC-Meter.p_sum3,Substation-AC-Meter.pln1,Substation-AC-Meter.pln2,Substation-AC-Meter.pln3,Substation-AC-Meter.q_sum3,Substation-AC-Meter.qln1,Substation-AC-Meter.qln2,Substation-AC-Meter.qln3,Substation-AC-Meter.s_sum3,Substation-AC-Meter.sln1,Substation-AC-Meter.sln2,Substation-AC-Meter.sln3,Substation-AC-Meter.ull12,Substation-AC-Meter.ull23,Substation-AC-Meter.ull31,Substation-AC-Meter.uln1,Substation-AC-Meter.uln2,Substation-AC-Meter.uln3
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.0,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.171057,0.0,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.171057,0.0,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.171057,0.0,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.505094,0.702562,0.972942,0.006344,0.005332,0.020741,0.018708,0.007502,41.564302,14.329292,13.174315,14.060694,120.026718,35.046867,49.145604,35.834249,1.333649,3.671471,2.083855,0.253975,0.01712,0.040283,0.095825,0.019806,0.008377,0.07695,0.171057

In [55]:
def drop_primeras(df, n):
    return df.iloc[n:].reset_index(drop=True)
dfsX={k: drop_primeras(df,4) for k, df in dfsX.items()}
dfsY={k: drop_primeras(df,4) for k, df in dfsY.items()}
print(dfsX["S1"].head(5),dfsY["S1"].head(5))

   BSS-Inverter-AC-Meter.cos_phi1  BSS-Inverter-AC-Meter.cos_phi2  BSS-Inverter-AC-Meter.cos_phi3  BSS-Inverter-AC-Meter.freq  BSS-Inverter-AC-Meter.i_sum3  \
0                        0.946213                        0.441541                       -0.228936                   49.988918                      0.117801   
1                        0.946213                        0.441541                       -0.228936                   49.988918                      0.117801   
2                        0.946213                        0.441541                       -0.228936                   49.988918                      0.117801   
3                        0.946213                        0.441541                       -0.228936                   49.988918                      0.117801   
4                       -0.018978                       -0.137548                       -0.240023                   49.991474                      0.095695   

   BSS-Inverter-AC-Meter.iln1  BSS-Inverter-A

In [51]:
# --- ubica el primer ataque sostenido en S3 y compara nivel crudo contra rango movil ---

def primer_ataque_sostenido(etiquetas, minimo=60):
    """Posicion del primer ataque que dura al menos `minimo` filas seguidas."""
    racha = 0
    for i, valor in enumerate(etiquetas):
        racha = racha + 1 if valor else 0
        if racha >= minimo:
            return i - minimo + 1
    return None


senal = "BSS-Inverter-AC-Meter.p_sum3"
escenario = "S1"

# --- series del escenario elegido: etiqueta, nivel crudo y rango movil ---
y_s3 = dfsY[escenario].to_numpy()
crudo = dfsX[escenario][senal].to_numpy()
rango = dfs2[escenario][senal].to_numpy()

inicio_rel = primer_ataque_sostenido(y_s3)
if inicio_rel is None:
    raise ValueError(f"sin ataque sostenido de 60 filas en {escenario}")

# --- ventana de 240 filas antes y 480 despues, recortada a los limites del escenario ---
inicio = max(inicio_rel - 240, 0)
fin = min(inicio_rel + 480, len(y_s3))
ventana = np.arange(inicio, fin)
x = ventana - inicio_rel
ataque = y_s3[ventana] == 1

fig, ejes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ejes[0].plot(x, crudo[ventana], lw=0.8)
ejes[0].set_title("BSS p_sum3: nivel crudo")
ejes[1].plot(x, rango[ventana], lw=0.8, color="tab:orange")
ejes[1].set_title("rango movil 120 s")
ejes[1].set_xlabel("segundos desde el inicio del ataque")

# --- sombrea solo hasta la ultima fila con attack_active = 1 dentro de la ventana ---
fin_ataque = x[ataque].max() if ataque.any() else 0
for eje in ejes:
    eje.axvspan(0, fin_ataque, color="tab:red", alpha=0.12, label="attack_active = 1")
    eje.axvline(0, color="tab:red", lw=1)
    eje.set_ylabel("W")
ejes[0].legend(loc="upper left")

# --- mediana del rango antes (x < 0) y durante el ataque, en la ventana ---
antes = x < 0
print("rango mediana antes del ataque:", round(float(np.nanmedian(rango[ventana][antes])), 1))
print("rango mediana durante el ataque:", round(float(np.nanmedian(rango[ventana][ataque])), 1))

fig.tight_layout()
plt.show()


KeyError: 'BSS-Inverter-AC-Meter.p_sum3'

In [52]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             f1_score, precision_score, recall_score)

PARAMS_RF = {
    "n_estimators": 120,
    "max_depth": 18,
    "min_samples_leaf": 2,
    "class_weight": "balanced",
    "random_state": 42,
    "n_jobs": -1,
}

escenarios = list(dfsX)
resultados = []
predicciones = {}

# --- un fold por escenario: entrena con los demas, predice el que queda afuera ---
for test in escenarios:
    X_train = pd.concat([dfsX[k] for k in escenarios if k != test], ignore_index=True)
    y_train = pd.concat([dfsY[k] for k in escenarios if k != test], ignore_index=True)
    X_test = dfsX[test].to_numpy()
    y_test = dfsY[test].to_numpy()

    rf = RandomForestClassifier(**PARAMS_RF).fit(X_train, y_train)
    pred = rf.predict(X_test)
    predicciones[test] = (y_test, pred)

    resultados.append({
        "escenario": test,
        "n": len(y_test),
        "positivos": int(y_test.sum()),
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
    })

tabla_rf = pd.DataFrame(resultados).set_index("escenario").round(3)
print("RF LOSO: 120 arboles, depth 18, leaf 2, class_weight=balanced")
display(tabla_rf)

# --- metricas globales sobre las predicciones out-of-fold de los 7 folds ---
y_true = np.concatenate([predicciones[k][0] for k in escenarios])
y_pred = np.concatenate([predicciones[k][1] for k in escenarios])
print("global accuracy:", round(accuracy_score(y_true, y_pred), 3))
print("global confusion matrix [[TN FP], [FN TP]]:")
print(confusion_matrix(y_true, y_pred))

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


RF LOSO: 120 arboles, depth 18, leaf 2, class_weight=balanced


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


,n,positivos,accuracy,precision,recall,f1
escenario,,,,,,
S1,1798,913,0.827,0.798,0.883,0.838
S1-alt,1792,910,0.748,0.668,1.000,0.801
S2,1768,903,0.615,0.681,0.464,0.552
S4,1798,1257,0.651,0.976,0.514,0.673
S5,1800,1341,0.653,0.941,0.570,0.710
S6,1800,1350,0.603,0.979,0.481,0.645


global accuracy: 0.683
global confusion matrix [[TN FP], [FN TP]]:
[[3152  930]
 [2479 4195]]


In [57]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             f1_score, precision_score, recall_score)

PARAMS_RF = {
    "n_estimators": 120,
    "max_depth": 18,
    "min_samples_leaf": 2,
    "class_weight": "balanced",
    "random_state": 42,
    "n_jobs": -1,
}

escenarios = list(dfsX)
resultados = []
predicciones = {}

# --- un fold por escenario: entrena con los demas, predice el que queda afuera ---
for test in escenarios:
    X_train = pd.concat([dfsX[k] for k in escenarios if k != test], ignore_index=True)
    y_train = pd.concat([dfsY[k] for k in escenarios if k != test], ignore_index=True)
    X_test = dfsX[test].to_numpy()
    y_test = dfsY[test].to_numpy()

    rf = RandomForestClassifier(**PARAMS_RF).fit(X_train, y_train)
    pred = rf.predict(X_test)
    predicciones[test] = (y_test, pred)

    resultados.append({
        "escenario": test,
        "n": len(y_test),
        "positivos": int(y_test.sum()),
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
    })

tabla_rf = pd.DataFrame(resultados).set_index("escenario").round(3)
print("RF LOSO: 120 arboles, depth 18, leaf 2, class_weight=balanced")
display(tabla_rf)

# --- metricas globales sobre las predicciones out-of-fold de los 7 folds ---
y_true = np.concatenate([predicciones[k][0] for k in escenarios])
y_pred = np.concatenate([predicciones[k][1] for k in escenarios])
print("global accuracy:", round(accuracy_score(y_true, y_pred), 3))
print("global confusion matrix [[TN FP], [FN TP]]:")
print(confusion_matrix(y_true, y_pred))

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


RF LOSO: 120 arboles, depth 18, leaf 2, class_weight=balanced


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


,n,positivos,accuracy,precision,recall,f1
escenario,,,,,,
S1,1794,913,0.818,0.802,0.852,0.826
S1-alt,1788,910,0.743,0.664,1.000,0.798
S2,1764,903,0.611,0.681,0.453,0.544
S4,1794,1257,0.665,0.971,0.538,0.692
S5,1796,1341,0.657,0.942,0.576,0.715
S6,1796,1350,0.600,0.974,0.481,0.644


global accuracy: 0.682
global confusion matrix [[TN FP], [FN TP]]:
[[3129  929]
 [2479 4195]]
